# Detecção e classificação de crimes cibernéticos em redes sociais utilizando algoritmos de aprendizado de máquina

**Alunos:**

Caio Vinícius Cunha Bernardo - *Nro. USP: 14652143*

Lucas Neves Souza - *Nro. USP: 15577627*

Willian Jefferson Sousa Farias - *Nro. USP: 15642665*

---

## Notebook 3 de 3: simulação dos resultados de Oliveira et al. (2024)

Reúne as tentativas de reproduzir, com **BERTimbau-Base em estágio único** (sem SVM), os resultados do artigo de referência, e de compará-los com a cascata dos notebooks 1 e 2.

**Como ler os experimentos**

- **Parte B (2 rótulos):** um classificador por rótulo, treinado no treino completo, com número fixo de épocas (3; nas nossas execuções, o *overfitting* aparece após a terceira época) e sem seleção no teste.
- **Parte C (13 rótulos):** mesmo treino (85%) e validação (15%) da cascata, mesma perda ponderada e mesmo critério de parada; a única diferença é a ausência do SVM e da amostragem 1:1.
- As comparações com a cascata usam os arquivos de `resultados/` gravados pelos notebooks 1 e 2.

**Projeto em três notebooks**

1. `1_Dois_Rotulos.ipynb`: análise exploratória e cascata (SVM + BERTimbau) no dataset **Binary** (`aggressive`, `hate`).
2. `2_Treze_Rotulos.ipynb`: análise exploratória e cascata no dataset **Multilabel** (13 rótulos), com ROC-AUC, matrizes de confusão, atenção e ablação da perda ponderada.
3. `3_Simulacao_Oliveira.ipynb` **(este notebook)**: baselines de estágio único para reproduzir os resultados de Oliveira et al. (2024) e compará-los com a cascata.

Os notebooks 1 e 2 gravam os resultados da cascata na pasta `resultados/`, que o notebook 3 lê para as comparações. Rode na ordem 1 → 2 → 3; o notebook 3 também roda sozinho, e nesse caso as comparações com a cascata são ignoradas.

# Setup

Primeiro, instalamos as bibliotecas do ecossistema **Hugging Face** que não vêm pré-instaladas no Google Colab. O comando `!pip` roda o `pip` no ambiente do Colab, a partir de uma célula do próprio notebook:

1) **transformers**: modelos pré-treinados (como o BERTimbau), *tokenizers* e as classes `Trainer` e `TrainingArguments`, que conduzem o ajuste fino (*fine-tuning*);
2) **datasets**: a estrutura `Dataset`, usada para armazenar e tokenizar os textos;
3) **accelerate**: biblioteca que executa o treinamento em PyTorch de forma eficiente no hardware disponível (GPU, precisão mista), exigida pelo `Trainer`.

O Hugging Face é a plataforma de código aberto onde desenvolvedores compartilham modelos de aprendizado de máquina, *datasets* e outras ferramentas de IA. As demais bibliotecas usadas aqui (NumPy, pandas, SciPy, scikit-learn e PyTorch) já vêm instaladas no Colab.

In [ ]:
!pip install -q transformers accelerate datasets

Importamos as bibliotecas usadas neste notebook:

1) **NumPy** e **pandas**: manipulação de arrays e tabelas (carregar o TuPy-E e montar as tabelas de resultados);
2) **scikit-learn**: usado apenas para as métricas (`classification_report`, F1, precisão e revocação) e para recriar a mesma divisão treino/validação do Notebook 2 (`train_test_split`). Os baselines de estágio único **não têm Fase 1**, então TF-IDF e SVM não são usados neste notebook;
3) **PyTorch**: tensores, uso da GPU e a perda BCE ponderada do baseline de 13 rótulos;
4) **Hugging Face**: `Dataset`, *tokenizer*, o modelo `AutoModelForSequenceClassification`, `DataCollatorWithPadding`, `Trainer`/`TrainingArguments`, `EarlyStoppingCallback` e `set_seed`.

Embora o pandas seja excelente para manipular tabelas e inspecionar os textos, o treinamento de redes neurais profundas trabalha com tensores. A biblioteca **datasets** converte os DataFrames do pandas para o formato `Dataset` (baseado em Apache Arrow, com mapeamento em memória), que permite tokenizar os textos em lotes com `.map` e se integra diretamente ao `Trainer`. O envio dos tensores à GPU ocorre durante o treinamento.

Além disso, no *deep learning* o treinamento ocorre em épocas e exige cálculo da perda, ajuste da taxa de aprendizado e controle do tamanho do lote. As classes **Trainer** e **TrainingArguments** encapsulam essa engenharia para que possamos ajustar o BERT com poucas linhas.

In [ ]:
import numpy as np
import pandas as pd

import torch
from datasets import Dataset

# Scikit-learn: métricas e divisão treino/validação (não há SVM/TF-IDF neste notebook)
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

# Hugging Face (modelo, tokenizer e treinamento) e PyTorch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
set_seed(SEED)

# Verificar se a GPU foi reconhecida pelo PyTorch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Processamento rodando em: {device}")

Processamento rodando em: cuda


O fluxo deste notebook é de **estágio único, sem triagem**: cada texto vai direto ao BERTimbau, ajustado com o `Trainer`. Na Parte B treinamos um classificador por rótulo (2 rótulos); na Parte C, um único modelo multirrótulo com 13 saídas sigmoides. O **pandas** e o **Dataset** organizam e tokenizam os textos, e o **scikit-learn** calcula as métricas.

A cascata (SVM + BERTimbau) dos Notebooks 1 e 2 **não é treinada aqui**: seus resultados são lidos dos arquivos gravados na pasta de resultados e usados apenas nas comparações.

Por fim, definimos o modelo pré-treinado (BERTimbau, versão *base*, a que reproduzimos; Oliveira et al. (2024) também avaliaram a versão *large* e o GPT-2, que não reproduzimos) e carregamos seu *tokenizer*, usado nos dois baselines:

In [ ]:
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_NAME)

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

# Importando Dataset pelo GitHub

In [ ]:
# Colunas que não serão úteis para o treinamento (rede social, pesquisador e ano)
COLUNAS_DESCARTADAS = ['source', 'researcher', 'year']

BASE_URL = ("https://raw.githubusercontent.com/Silly-Machine/"
            "TuPyE-Expanded-Brazilian-Hate-Speech-Dataset/refs/heads/master")

# Binary
df_binary_train = pd.read_csv(f"{BASE_URL}/binary/binary_train.csv")
df_binary_test = pd.read_csv(f"{BASE_URL}/binary/binary_test.csv")
# Multilabel
df_multilabel_train = pd.read_csv(f"{BASE_URL}/multilabel/multilabel_train.csv")
df_multilabel_test = pd.read_csv(f"{BASE_URL}/multilabel/multilabel_test.csv")

## Preparação dos dados

In [ ]:
# Dataset Binary (2 rótulos)
COLUNAS_2 = ['aggressive', 'hate']
df_binary_train_cleaned = df_binary_train.drop(columns=COLUNAS_DESCARTADAS)
df_binary_test_cleaned = df_binary_test.drop(columns=COLUNAS_DESCARTADAS)
y_true_total = df_binary_test_cleaned[COLUNAS_2].values.astype(int)

# Dataset Multilabel (13 rótulos)
COLUNAS_13 = ['aggressive', 'hate', 'ageism', 'aporophobia', 'body_shame', 'capacitism', 'lgbtphobia',
 'political', 'racism', 'religious_intolerance', 'misogyny', 'xenophobia', 'other']
df_multilabel_train_cleaned = df_multilabel_train.drop(columns=COLUNAS_DESCARTADAS)
df_multilabel_test_cleaned = df_multilabel_test.drop(columns=COLUNAS_DESCARTADAS)
# 'label' = qualquer um dos 13 rótulos; usado só para recriar a mesma divisão treino/validação do Notebook 2
df_multilabel_train_cleaned['label'] = df_multilabel_train_cleaned[COLUNAS_13].max(axis=1)

# Tokenização usada nos baselines (texto bruto, truncamento em 128 tokens; padding dinâmico por lote)
def tokenizar(exemplos):
    return tokenizer_bert(exemplos['text'], truncation=True, max_length=128)

## Funções auxiliares

In [ ]:
from scipy import stats
from sklearn.metrics import precision_recall_fscore_support, f1_score

def relatorio_df(y_true, y_pred, nomes):
    rep = classification_report(y_true, y_pred, target_names=nomes,
                                output_dict=True, zero_division=0)
    acc = rep.pop('accuracy', None)
    df = pd.DataFrame(rep).T.rename(columns={'precision': 'Precisão', 'recall': 'Revocação',
                                             'f1-score': 'F1', 'support': 'Suporte'})
    if acc is not None:   # só existe quando o rótulo é único
        df.loc['Acurácia global'] = [np.nan, np.nan, acc, df.loc['macro avg', 'Suporte']]
    return df

def estilizar(df, titulo):
    return (df.style.set_caption(titulo)
              .format({'Precisão': '{:.2f}', 'Revocação': '{:.2f}', 'F1': '{:.2f}',
                       'Suporte': '{:.0f}'}, na_rep='–')
              .background_gradient(cmap='Blues', subset=['Precisão', 'Revocação', 'F1'], vmin=0, vmax=1))

def metricas_oliveira(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted', zero_division=0)
    return {'Precisão (pond.)': p, 'Revocação (pond.)': r, 'F1 (pond.)': f,
            'F1 macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
            'F1 classe positiva': f1_score(y_true, y_pred, zero_division=0)}

## Resultados da cascata (notebooks 1 e 2)

Verifica se os arquivos gravados pelos notebooks 1 e 2 estão disponíveis. Sem eles, as comparações com a cascata são ignoradas.

In [ ]:
import os
from google.colab import drive

# 1. Monta o Google Drive no ambiente do Colab
drive.mount('/content/drive')

# 2. Aponta para a mesma pasta usada nos notebooks 1 e 2
PASTA_RESULTADOS = "/content/drive/MyDrive/Resultados_crimescib"

ARQ_CASCATA_2 = os.path.join(PASTA_RESULTADOS, "cascata_2rotulos.npz")
ARQ_CASCATA_13 = os.path.join(PASTA_RESULTADOS, "cascata_13rotulos.npz")

TEM_CASCATA_2 = os.path.exists(ARQ_CASCATA_2)
TEM_CASCATA_13 = os.path.exists(ARQ_CASCATA_13)

print("Cascata (2 rótulos) :", "encontrada" if TEM_CASCATA_2 else "não encontrada (rode o Notebook 1)")
print("Cascata (13 rótulos):", "encontrada" if TEM_CASCATA_13 else "não encontrada (rode o Notebook 2)")

Mounted at /content/drive
Cascata (2 rótulos) : encontrada
Cascata (13 rótulos): encontrada


# Parte A: referência de Oliveira et al. e cascata (2 rótulos)

Métricas no formato do artigo de referência (precisão, revocação e F1 ponderados), para a cascata ponta a ponta (com sensibilidade ao limiar do SVM, sem retreinar) e para os valores reportados por Oliveira et al. (2024), Seção 4.1.

In [ ]:
linhas = {}

if TEM_CASCATA_2:
    dados2 = np.load(ARQ_CASCATA_2)
    assert np.array_equal(dados2['y_true_total'], y_true_total), "O teste salvo no Notebook 1 não coincide com o atual"
    pred_bert_total = dados2['pred_bert_total']
    probabilidade_crime_svm = dados2['probabilidade_crime_svm']
    limiar_svm_calibrado = float(dados2['limiar_svm_calibrado'])

    def cascata(limiar):
        passou = (probabilidade_crime_svm >= limiar)[:, None]
        return pred_bert_total * passou

    # sensibilidade ao limiar, sem retreinar (só limiares >= o escolhido)
    for limiar in sorted({limiar_svm_calibrado, *[l for l in (0.30, 0.50) if l > limiar_svm_calibrado]}):
        yp = cascata(limiar)
        linhas[f'Cascata hate (SVM {limiar})'] = metricas_oliveira(y_true_total[:, 1], yp[:, 1])
        linhas[f'Cascata aggressive (SVM {limiar})'] = metricas_oliveira(y_true_total.max(1), yp.max(1))
else:
    print("Cascata não encontrada: apenas a referência de Oliveira será exibida.")

# Referência: Seção 4.1 de Oliveira et al. (2024)
ref = {'BERTimbau-Base': (0.897, 0.901, 0.899), 'BERTimbau-Large': (0.901, 0.907, 0.903),
       'GPT-2 Small': (0.888, 0.892, 0.890)}
for nome, (p, r, f) in ref.items():
    linhas[f'{nome} (Oliveira, estágio único)'] = {'Precisão (pond.)': p, 'Revocação (pond.)': r, 'F1 (pond.)': f}

display(pd.DataFrame(linhas).T.round(3))

,Precisão (pond.),Revocação (pond.),F1 (pond.),F1 macro,F1 classe positiva
Cascata hate (SVM 0.12000000000000001),0.896,0.893,0.894,0.753,0.567
Cascata aggressive (SVM 0.12000000000000001),0.839,0.830,0.833,0.789,0.696
Cascata hate (SVM 0.3),0.892,0.900,0.895,0.742,0.540
Cascata aggressive (SVM 0.3),0.830,0.836,0.831,0.776,0.661
Cascata hate (SVM 0.5),0.885,0.899,0.887,0.705,0.466
Cascata aggressive (SVM 0.5),0.817,0.823,0.807,0.734,0.580
"BERTimbau-Base (Oliveira, estágio único)",0.897,0.901,0.899,NaN,NaN
"BERTimbau-Large (Oliveira, estágio único)",0.901,0.907,0.903,NaN,NaN
"GPT-2 Small (Oliveira, estágio único)",0.888,0.892,0.890,NaN,NaN


# Parte B: baseline de estágio único, 2 rótulos

Um BERTimbau por rótulo (`hate` e `aggressive`), treinado no treino completo, sem SVM e sem amostragem de benignos. O número de épocas é fixo e a avaliação usa o teste uma única vez, sem seleção de época.

In [ ]:
def treinar_baseline(coluna, epocas=10, seed=SEED):
    set_seed(seed)
    tr = df_binary_train_cleaned[['text', coluna]].rename(columns={coluna: 'label'})
    te = df_binary_test_cleaned[['text', coluna]].rename(columns={coluna: 'label'})
    ds_tr = Dataset.from_pandas(tr, preserve_index=False).map(tokenizar, batched=True)
    ds_te = Dataset.from_pandas(te, preserve_index=False).map(tokenizar, batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    args = TrainingArguments(output_dir=f"./baseline_{coluna}", num_train_epochs=epocas,
                             learning_rate=1e-5, per_device_train_batch_size=32,
                             warmup_steps=500, weight_decay=0.01, save_strategy="no",
                             eval_strategy="no", report_to="none", seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=ds_tr,
                      data_collator=DataCollatorWithPadding(tokenizer=tokenizer_bert))
    trainer.train()
    return trainer.predict(ds_te).predictions.argmax(-1)

pred_base_hate = treinar_baseline('hate', epocas=3)         # teste rápido; depois use 10
pred_base_aggr = treinar_baseline('aggressive', epocas=3)

linhas_base = {
  'Baseline hate (nosso, estágio único)': metricas_oliveira(y_true_total[:, 1], pred_base_hate),
  'Baseline aggressive (nosso, estágio único)': metricas_oliveira(y_true_total[:, 0], pred_base_aggr),
}
display(pd.DataFrame(linhas_base).T.round(3))

Map:   0%|          | 0/34934 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

... (5 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Step,Training Loss
500,0.510355
1000,0.380568
1500,0.332976
2000,0.320292
2500,0.281055
3000,0.267474


,Precisão (pond.),Revocação (pond.),F1 (pond.),F1 macro,F1 classe positiva
"Baseline hate (nosso, estágio único)",0.899,0.905,0.901,0.759,0.571
"Baseline aggressive (nosso, estágio único)",0.847,0.851,0.848,0.800,0.700


# Parte C: baseline de estágio único, 13 rótulos

Mesma partição treino/validação da cascata (85%/15%, mesma semente e mesma estratificação, recriada de forma determinística), mesma perda ponderada e mesmo critério de parada. A diferença para a cascata é a ausência do SVM e da amostragem 1:1.

In [ ]:
from sklearn.model_selection import train_test_split

colunas_crimes = COLUNAS_13

# Mesma divisão treino/validação do Notebook 2 (determinística: mesma semente e mesma estratificação)
df_sub, df_val = train_test_split(df_multilabel_train_cleaned, test_size=0.15,
                                  stratify=df_multilabel_train_cleaned['label'], random_state=SEED)

if TEM_CASCATA_13:
    idx_val_salvo = np.load(ARQ_CASCATA_13)['idx_val']
    if not np.array_equal(np.sort(idx_val_salvo), np.sort(df_val.index.values)):
        print("⚠️ A validação recriada difere da usada no Notebook 2 (versões diferentes do scikit-learn?).")

In [11]:
# Baseline de estágio único com 13 rótulos (sem SVM e sem amostragem 1:1)
class PesoBCETrainer(Trainer):
    """Mesma perda ponderada da Fase 2, mas com pos_weight próprio (não altera o pos_weight global)."""
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pw = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = torch.nn.BCEWithLogitsLoss(pos_weight=self.pw)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

def para_dataset(df):
    d = df[['text']].copy()
    d['labels'] = df[colunas_crimes].values.astype(float).tolist()
    return Dataset.from_pandas(d, preserve_index=False).map(tokenizar, batched=True)

# Mesmo treino (85%) e validação (15%) da cascata, mas SEM filtro do SVM e SEM amostragem 1:1
ds_b_train, ds_b_val = para_dataset(df_sub), para_dataset(df_val)
ds_b_test = para_dataset(df_multilabel_test_cleaned)                # teste completo

Yb = df_sub[colunas_crimes].values
pos_b = Yb.sum(axis=0); neg_b = len(Yb) - pos_b
pw_b = torch.tensor(np.clip(neg_b / np.maximum(pos_b, 1), 1, 50), dtype=torch.float).to(device)

set_seed(SEED)
model_b = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(colunas_crimes), problem_type="multi_label_classification").to(device)

args_b = TrainingArguments(                      # mesmos hiperparâmetros da Fase 2 da cascata
    output_dir="./baseline_13rotulos", eval_strategy="epoch", save_strategy="epoch",
    learning_rate=2e-5, per_device_train_batch_size=16, num_train_epochs=5, weight_decay=0.01,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", seed=SEED, report_to="none")

trainer_b = PesoBCETrainer(
    model=model_b, args=args_b, train_dataset=ds_b_train, eval_dataset=ds_b_val,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer_bert),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)], pos_weight=pw_b)
trainer_b.train()

# Avaliação no teste completo (uma única vez)
logits_b = trainer_b.predict(ds_b_test).predictions
y_pred_base = (1 / (1 + np.exp(-logits_b)) >= 0.5).astype(int)

Y_total = df_multilabel_test_cleaned[colunas_crimes].values
display(estilizar(relatorio_df(Y_total, y_pred_base, colunas_crimes), 'Baseline de estágio único (13 rótulos)'))

if TEM_CASCATA_13:
    dados13 = np.load(ARQ_CASCATA_13)
    assert np.array_equal(dados13['Y_total'], Y_total), "O teste salvo no Notebook 2 não coincide com o atual"
    y_pred_casc = dados13['y_pred_casc']                                # barrado pelo SVM = 0

    comp13 = pd.concat({'Estágio único': relatorio_df(Y_total, y_pred_base, colunas_crimes),
                        'Cascata (ponta a ponta)': relatorio_df(Y_total, y_pred_casc, colunas_crimes)}, axis=1)
    cols_metricas = ['Precisão', 'Revocação', 'F1']
    comp13 = comp13.loc[colunas_crimes + ['macro avg'],
                        [(g, m) for g in ['Estágio único', 'Cascata (ponta a ponta)'] for m in cols_metricas]]
    display(comp13.style.format('{:.2f}').background_gradient(cmap='Blues', vmin=0, vmax=1))
else:
    print("Cascata (Notebook 2) não encontrada: comparação ignorada.")

Map:   0%|          | 0/29693 [00:00<?, ? examples/s]

Map:   0%|          | 0/5241 [00:00<?, ? examples/s]

Map:   0%|          | 0/8734 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

Epoch,Training Loss,Validation Loss
1,0.572277,0.527371


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss
1,0.572277,0.527371
2,0.385921,0.509371
3,0.283117,0.509498


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

,Precisão,Revocação,F1,Suporte
aggressive,0.64,0.79,0.70,2280
hate,0.53,0.72,0.61,1050
ageism,0.06,0.17,0.09,12
aporophobia,0.25,0.50,0.33,14
body_shame,0.31,0.68,0.43,63
capacitism,0.03,0.09,0.05,11
lgbtphobia,0.42,0.85,0.56,149
political,0.41,0.64,0.50,230
racism,0.19,0.64,0.29,56
religious_intolerance,0.14,0.47,0.21,17


## Teste de McNemar: cascata × estágio único (13 rótulos)

Compara as previsões dos dois modelos nos mesmos textos do teste completo, por rótulo e com correção de Holm. A = cascata, B = estágio único. Com os positivos reais, compara-se a revocação; com os negativos reais, os falsos positivos.

In [12]:
def mcnemar_exato(acerto_a, acerto_b):
    b01 = int(((acerto_a == 1) & (acerto_b == 0)).sum())   # só A acerta
    c01 = int(((acerto_a == 0) & (acerto_b == 1)).sum())   # só B acerta
    p = 1.0 if b01 + c01 == 0 else stats.binomtest(b01, b01 + c01, 0.5).pvalue
    return b01, c01, p

def holm(p):
    p = np.asarray(p, float); m = len(p); aj = np.empty(m); atual = 0
    for rank, i in enumerate(np.argsort(p)):
        atual = max(atual, (m - rank) * p[i]); aj[i] = min(1, atual)
    return aj

def mcnemar_por_rotulo(Y, pred_a, pred_b, nomes, sobre='positivos'):
    alvo = 1 if sobre == 'positivos' else 0
    linhas = []
    for j, nome in enumerate(nomes):
        m = Y[:, j] == alvo
        b01, c01, p = mcnemar_exato((pred_a[m, j] == Y[m, j]).astype(int),
                                    (pred_b[m, j] == Y[m, j]).astype(int))
        linhas.append({'Rótulo': nome, 'n': int(m.sum()), 'só A acerta': b01, 'só B acerta': c01, 'p': p})
    df = pd.DataFrame(linhas).set_index('Rótulo')
    df['p (Holm)'] = holm(df['p'].values)
    return df

In [13]:
if TEM_CASCATA_13:
    display(mcnemar_por_rotulo(Y_total, y_pred_casc, y_pred_base, colunas_crimes, 'positivos').round(3))  # compara revocação
    display(mcnemar_por_rotulo(Y_total, y_pred_casc, y_pred_base, colunas_crimes, 'negativos').round(3))  # compara falsos positivos
else:
    print("Cascata (Notebook 2) não encontrada: teste ignorado.")

,n,só A acerta,só B acerta,p,p (Holm)
Rótulo,,,,,
aggressive,2280,126,154,0.106,1.000
hate,1050,86,78,0.585,1.000
ageism,12,0,2,0.500,1.000
aporophobia,14,1,1,1.000,1.000
body_shame,63,1,7,0.070,0.773
capacitism,11,1,0,1.000,1.000
lgbtphobia,149,2,11,0.022,0.270
political,230,18,13,0.473,1.000
racism,56,6,6,1.000,1.000


,n,só A acerta,só B acerta,p,p (Holm)
Rótulo,,,,,
aggressive,6454,218,408,0.000,0.000
hate,7684,181,318,0.000,0.000
ageism,8722,15,6,0.078,0.313
aporophobia,8720,7,20,0.019,0.134
body_shame,8671,37,59,0.032,0.189
capacitism,8723,11,21,0.110,0.331
lgbtphobia,8585,101,31,0.000,0.000
political,8504,79,106,0.056,0.278
racism,8678,70,59,0.379,0.757


## Bootstrap pareado: cascata × estágio único

**Objetivo.** Medir quanta incerteza a amostragem do conjunto de teste coloca na diferença de F1 entre a cascata e o baseline de estágio único (Δ = cascata − baseline). Responde à pergunta central do trabalho: a diferença observada é maior do que o acaso da amostra explicaria?

**Como funciona.** Em cada uma das 1.000 reamostragens, sorteamos com reposição os mesmos textos do teste e aplicamos aos dois modelos (bootstrap *pareado*). Calculamos o F1 de cada um e a diferença. O IC de 95% é o intervalo entre os percentis 2,5 e 97,5 dessas 1.000 diferenças, e `P(Δ>0)` é a fração das reamostragens em que a cascata supera o baseline. Se o IC inclui 0, não há evidência de diferença.

**Esta célula** só define as funções (não imprime nada):
- `pesos_bootstrap`: gera, para cada reamostragem, quantas vezes cada texto entra.
- `f1_por_rotulo`: calcula o F1 de cada rótulo em todas as reamostragens de uma vez, a partir de verdadeiros positivos, falsos positivos e falsos negativos.
- `metricas_binarias`: F1 ponderado, macro e da classe positiva (as colunas da Tabela 4).
- `resumo_diferenca`: converte as 1.000 diferenças em Δ observado, mediana, IC de 95% e `P(Δ>0)`.

**Não retreina nada.** Usa as previsões já calculadas nas Partes B e C, então essas partes precisam ter rodado antes.

**Limitação.** Só captura a variação do conjunto de teste, não a do treino (cada modelo foi treinado com uma única semente).

In [14]:
# ============ Bootstrap pareado (1.000 reamostragens): cascata − estágio único ============
N_BOOT = 1000
RARAS = ['ageism', 'aporophobia', 'capacitism', 'religious_intolerance']

def pesos_bootstrap(n, B=N_BOOT, seed=SEED):
    """Quantas vezes cada texto entra em cada reamostragem: matriz (B, n), cada linha soma n.
    Equivale a sortear n textos com reposição; os mesmos pesos valem para os dois modelos (pareado)."""
    rng = np.random.default_rng(seed)
    return rng.multinomial(n, np.full(n, 1.0 / n), size=B).astype(np.float32)

def f1_por_rotulo(Y, P, W):
    """F1 de cada coluna (rótulo) em cada reamostragem. Y, P: (n, L) binários; W: (B, n).
    Retorna (f1, suporte), ambos (B, L). F1 = 0 quando não há positivos reais nem preditos (zero_division=0)."""
    Y = Y.astype(np.float32); P = P.astype(np.float32)
    tp = W @ (Y * P); fp = W @ ((1 - Y) * P); fn = W @ (Y * (1 - P))
    den = 2 * tp + fp + fn
    f1 = np.divide(2 * tp, den, out=np.zeros_like(den), where=den > 0)
    return f1, tp + fn

def metricas_binarias(y, p, W):
    """F1 ponderado, F1 macro (média das classes 0 e 1) e F1 da classe positiva, como na Tabela 4."""
    Y2 = np.column_stack([1 - y, y]); P2 = np.column_stack([1 - p, p])
    f1, sup = f1_por_rotulo(Y2, P2, W)
    return {'F1 ponderado': (f1 * sup).sum(1) / sup.sum(1),
            'F1 macro': f1.mean(1),
            'F1 positivo': f1[:, 1]}

def resumo_diferenca(obs, boot, alpha=0.05):
    lo, hi = np.percentile(boot, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return {'Δ observado': obs, 'Mediana (boot)': np.median(boot),
            'IC95% inf': lo, 'IC95% sup': hi, 'P(Δ>0)': (boot > 0).mean(),
            'IC inclui 0': 'sim' if lo <= 0 <= hi else 'não'}

### Bootstrap com 2 rótulos (`hate` e `aggressive`)

Compara a cascata de 2 rótulos (SVM com o limiar `limiar_svm_calibrado` + BERTimbau) com o baseline de estágio único da Parte B, usando as mesmas previsões da Tabela 4. A cascata é ponta a ponta: os textos barrados pelo SVM entram com previsão 0.

- Para `aggressive`, o rótulo verdadeiro é a coluna 0 e a previsão da cascata é o máximo entre as duas saídas. O `assert` confirma que todo texto de ódio também é agressivo.
- Antes de reamostrar, a célula confere (`assert`) que a estimativa observada bate com `metricas_oliveira` (sklearn). Se passar, os valores de *Cascata* e *Baseline* são os mesmos da Tabela 4.

**Como ler a saída.** Cada linha traz o F1 da cascata, o do baseline, o Δ observado e o IC de 95% da diferença. `IC inclui 0 = sim` significa que não há diferença estatisticamente distinguível. Para o artigo, indique qual F1 sustenta a conclusão (a coluna *F1 positivo* é a mais sensível à classe minoritária).

In [15]:
# ---------- 2 rótulos (mesmas previsões da Tabela 4) ----------
print("=== Bootstrap: 2 rótulos (Δ = cascata − baseline) ===")
if not TEM_CASCATA_2:
    print("Cascata (2 rótulos) não encontrada: bootstrap binário ignorado.")
elif 'pred_base_hate' not in globals():
    print("Rode antes a Parte B (baseline de 2 rótulos).")
else:
    n = len(y_true_total)
    assert np.array_equal(y_true_total.max(1), y_true_total[:, 0]), "hate deveria ser subconjunto de aggressive"
    W = pesos_bootstrap(n)
    W1 = np.ones((1, n), dtype=np.float32)          # "reamostragem" identidade = estimativa observada
    yp_casc = cascata(limiar_svm_calibrado)
    comparacoes = {'hate':       (y_true_total[:, 1], yp_casc[:, 1],   pred_base_hate),
                   'aggressive': (y_true_total[:, 0], yp_casc.max(1), pred_base_aggr)}
    linhas = []
    for nome, (y, p_c, p_b) in comparacoes.items():
        obs_c, obs_b = metricas_binarias(y, p_c, W1), metricas_binarias(y, p_b, W1)
        # confere a implementação vetorizada contra o sklearn (mesmos números da Tabela 4)
        ref = metricas_oliveira(y, p_c)
        assert abs(obs_c['F1 ponderado'][0] - ref['F1 (pond.)']) < 1e-5
        assert abs(obs_c['F1 macro'][0] - ref['F1 macro']) < 1e-5
        assert abs(obs_c['F1 positivo'][0] - ref['F1 classe positiva']) < 1e-5
        boot_c, boot_b = metricas_binarias(y, p_c, W), metricas_binarias(y, p_b, W)
        for m in boot_c:
            linhas.append({'Rótulo': nome, 'Métrica': m,
                           'Cascata': obs_c[m][0], 'Baseline': obs_b[m][0],
                           **resumo_diferenca(obs_c[m][0] - obs_b[m][0], boot_c[m] - boot_b[m])})
    tab_boot_bin = pd.DataFrame(linhas).set_index(['Rótulo', 'Métrica'])
    display(tab_boot_bin.round(3))

=== Bootstrap: 2 rótulos (Δ = cascata − baseline) ===


Cascata  Baseline  Δ observado  Mediana (boot)  \
Rótulo     Métrica                                                        
hate       F1 ponderado    0.894     0.901       -0.007          -0.007   
           F1 macro        0.753     0.759       -0.006          -0.005   
           F1 positivo     0.567     0.571       -0.004          -0.003   
aggressive F1 ponderado    0.833     0.848       -0.015          -0.015   
           F1 macro        0.789     0.800       -0.011          -0.011   
           F1 positivo     0.696     0.700       -0.003          -0.004   

                         IC95% inf  IC95% sup  P(Δ>0) IC inclui 0  
Rótulo     Métrica                                                 
hate       F1 ponderado     -0.012     -0.002   0.002         não  
           F1 macro         -0.018      0.006   0.210         sim  
           F1 positivo      -0.026      0.018   0.403         sim  
aggressive F1 ponderado     -0.022     -0.009   0.000         não  
           F1 macro         -0.019     -0.003   0.002         não  
           F1 positivo      -0.016      0.008   0.275         sim

### Bootstrap com 13 rótulos (cascata ponta a ponta × baseline)

Compara a cascata de 13 rótulos (`y_pred_casc`, que inclui os erros da Fase 1) com o BERTimbau de estágio único da Parte C (`y_pred_base`), no conjunto de teste completo.

A tabela tem:
- **F1 macro (13 rótulos):** média do F1 dos 13 rótulos, comparável ao F1 macro de Oliveira et al. (2024).
- **F1 macro (4 raras):** média do F1 de etarismo, aporofobia, capacitismo e intolerância religiosa. É o ponto central da pergunta de pesquisa.
- **F1 por rótulo**, com o suporte (número de textos positivos no teste).

Os `assert` conferem que o F1 por rótulo calculado aqui é igual ao do sklearn antes de reamostrar.

**Como ler.** Cuidado com as classes raras: o suporte é de poucas dezenas de textos, então os ICs são largos e quase sempre incluem 0. Os ICs por rótulo não têm correção para múltiplas comparações (diferente do McNemar com Holm), então use-os como descrição da incerteza, não como testes de hipótese.

In [16]:
# ---------- 13 rótulos (cascata ponta a ponta × baseline de estágio único) ----------
print("=== Bootstrap: 13 rótulos (Δ = cascata − baseline) ===")
if not TEM_CASCATA_13:
    print("Cascata (13 rótulos) não encontrada: bootstrap multirrótulo ignorado.")
elif 'y_pred_base' not in globals():
    print("Rode antes a Parte C (baseline de 13 rótulos).")
else:
    n = len(Y_total)
    W = pesos_bootstrap(n)
    W1 = np.ones((1, n), dtype=np.float32)
    idx_raras = [colunas_crimes.index(c) for c in RARAS]

    f1_c, _  = f1_por_rotulo(Y_total, y_pred_casc, W)
    f1_b, _  = f1_por_rotulo(Y_total, y_pred_base, W)
    f1_c1, _ = f1_por_rotulo(Y_total, y_pred_casc, W1)
    f1_b1, _ = f1_por_rotulo(Y_total, y_pred_base, W1)

    # confere contra o sklearn (F1 por rótulo, estimativa observada)
    assert np.allclose(f1_c1[0], f1_score(Y_total, y_pred_casc, average=None, zero_division=0), atol=1e-5)
    assert np.allclose(f1_b1[0], f1_score(Y_total, y_pred_base, average=None, zero_division=0), atol=1e-5)

    agregados = {'F1 macro (13 rótulos)': lambda f: f.mean(1),
                 'F1 macro (4 raras)':    lambda f: f[:, idx_raras].mean(1)}
    linhas = []
    for nome, agg in agregados.items():
        oc, ob = agg(f1_c1)[0], agg(f1_b1)[0]
        linhas.append({'Métrica': nome, 'Suporte': np.nan, 'Cascata': oc, 'Baseline': ob,
                       **resumo_diferenca(oc - ob, agg(f1_c) - agg(f1_b))})
    for j, rotulo in enumerate(colunas_crimes):
        linhas.append({'Métrica': f'F1 {rotulo}', 'Suporte': int(Y_total[:, j].sum()),
                       'Cascata': f1_c1[0, j], 'Baseline': f1_b1[0, j],
                       **resumo_diferenca(f1_c1[0, j] - f1_b1[0, j], f1_c[:, j] - f1_b[:, j])})
    tab_boot_13 = pd.DataFrame(linhas).set_index('Métrica')
    display(tab_boot_13.round(3))

=== Bootstrap: 13 rótulos (Δ = cascata − baseline) ===


,Suporte,Cascata,Baseline,Δ observado,Mediana (boot),IC95% inf,IC95% sup,P(Δ>0),IC inclui 0
Métrica,,,,,,,,,
F1 macro (13 rótulos),NaN,0.379,0.396,-0.016,-0.016,-0.034,0.002,0.038,sim
F1 macro (4 raras),NaN,0.142,0.169,-0.027,-0.026,-0.072,0.022,0.137,sim
F1 aggressive,2280.0,0.671,0.704,-0.032,-0.033,-0.043,-0.022,0.000,não
F1 hate,1050.0,0.580,0.608,-0.027,-0.027,-0.045,-0.011,0.001,não
F1 ageism,12.0,0.000,0.085,-0.085,-0.080,-0.200,0.000,0.000,sim
F1 aporophobia,14.0,0.255,0.333,-0.079,-0.077,-0.198,0.035,0.096,sim
F1 body_shame,63.0,0.341,0.428,-0.087,-0.087,-0.142,-0.032,0.001,não
F1 capacitism,11.0,0.073,0.045,0.027,0.025,-0.025,0.113,0.620,sim
F1 lgbtphobia,149.0,0.636,0.564,0.072,0.070,0.031,0.113,1.000,não
